In [21]:
import fitz
import json
import os
import pandas as pd
import re
from collections import Counter

### Hitung jumlah kamus dan kamus duplicate

In [22]:
from collections import defaultdict

folder_path = '/Users/rafaeldewandaru/SEMESTER 8/Tugas Akhir/PDF'

pdf_files = [f for f in os.listdir(folder_path) if f.lower().endswith(".pdf")]

print(f"Total kamus: {len(pdf_files)}")
print("==================================")


groups = defaultdict(list)

for f in pdf_files:
    prefix = f.split('.')[0].strip()

    match = re.match(r'\d+', prefix)
    if match:
        number = match.group()
        groups[number].append(f)

print("Duplicate kamus:")

for number, files in sorted(groups.items()):
    if len(files) > 1:
        for f in files:
            print(f)
        print()

Total kamus: 99
Duplicate kamus:
88. Kamus Sasak-Indonesia ed2 (2017).pdf
88b. Kamus Sasak-Indonesia ed2 (2018).pdf

89b. Kamus Dwibahasa Bahasa Mooi-Bahasa Indonesia (2017).pdf
89. Kamus Dwibahasa Bahasa Mooi-Bahasa Indonesia (2017).pdf

90. Kamus Mbojo-Indonesia (2015).pdf
90b. Kamus Mbojo-Indonesia (2015).pdf

91. Kamus Simalungun - Indonesia (edisi kedua) (2015).pdf
91b. Kamus Bahasa Simalungun-Indonesia ed2 (2016).pdf



### Hitung jumlah kamus yang berhasil dikonversi ke bentuk JSON
output dari kode Harakan *1. Konversi PDF to JSON.ipynb*

In [23]:
import json

folder_path = "/Users/rafaeldewandaru/SEMESTER 8/Tugas Akhir/TAEkstraksiKamus/[Full] Bentuk JSON"

valid_count = 0
invalid_files = []

for filename in os.listdir(folder_path):
    if filename.lower().endswith(".json"):
        filepath = os.path.join(folder_path, filename)

        try:
            with open(filepath, "r", encoding="utf-8") as f:
                json.load(f)
            valid_count += 1
        except json.JSONDecodeError:
            invalid_files.append(filename)

print("Valid JSON files:", valid_count)
print("Invalid JSON files:", len(invalid_files))

if invalid_files:
    print("\nInvalid files:")
    for f in invalid_files:
        print(f)

Valid JSON files: 96
Invalid JSON files: 3

Invalid files:
76. Kamus kemaritiman Aceh - Indonesia (2021)-hasil.json
70. Kamus dwibahasa Hitu - Indonesia (2017)-hasil.json
89. Kamus Dwibahasa Bahasa Mooi-Bahasa Indonesia (2017)-hasil.json


#### Key takeaways:
Kamus **89. Kamus Dwibahasa Bahasa Mooi-Bahasa Indonesia (2017)** yang seharusnya menjadi sample untuk penulisan entri Homonym, Synonym, dan Polysemy, gagal konversi PDF ke JSON. 

Namun untuk kamus **89b. Kamus Dwibahasa Bahasa Mooi-Bahasa Indonesia (2017)** berhasil dikonversi menjadi JSON

### Hitung jumlah kamus layak untuk ekstrak informasi JSON
output dari kode Harakan *2. Ekstrak Informasi JSON.ipynb*

Terdapat 2 kali proses filtering dalam dengan output
1. [Full] Raw CSV JSON all information

2. [Full] CSV JSON all information - Final

In [24]:
folder_path = '/Users/rafaeldewandaru/SEMESTER 8/Tugas Akhir/TAEkstraksiKamus/[Full] Raw CSV JSON all information'

count = len(os.listdir(folder_path))
print(f"Total kamus layak setelah filter 1: {count}")

Total kamus layak setelah filter 1: 76


Filter dilakukan supaya hanya melakukan ekstraksi informasi JSON untuk kamus yang memiliki font Bold dan Italic. Sebagai perbandingan, jumlah output dari Harakan adalah sebanyak 72 kamus.

In [25]:
folder_path = '/Users/rafaeldewandaru/SEMESTER 8/Tugas Akhir/TAEkstraksiKamus/[Full] CSV JSON all information - Final'

count = len(os.listdir(folder_path))
print(f"Total kamus layak setelah filter 2: {count}")

Total kamus layak setelah filter 2: 58


Dari hasil filter yang sebelumnya, dilakukan pembuangan halaman yang tidak relevan dengan entri seperti halaman judul, kata pengantar, dll. yang kemudian dilakukan filter lagi yang masih memiliki font Bold. Sebagai perbadingan, jumlah output dari Harakan adalah sebanyak 61 kamus.

### Pembentukan korpus entri

Untuk menentukan kamus mana saja yang dibentuk korpus entri nya, dilakukan exploratory kamus terlebih dahulu untuk melihat jumlah kata bold dan kata italic yang terdapat dalam kamus. Korpus entri akan dibentuk untuk kamus yang memenuhi kriteria berikut

**jumlah_bold >= banyak_halaman & jumlah_italic >= banyak_halaman**

In [27]:
kamus_ekstraksi = "/Users/rafaeldewandaru/SEMESTER 8/Tugas Akhir/TAEkstraksiKamus/Harakan/daftar_kamus_ekstraks.csv"
pandas_res = pd.read_csv(kamus_ekstraksi)

display(pandas_res)

,nama_kamus,jumlah_bold,jumlah_italic,mean_ukuran_char,banyak_halaman
0,27. Kamus Bahasa Indonesia-Saluan (2012)-hasil...,9105,11147,10.10,120
1,63. Kamus Bahasa Indonesia-Lampung Dialek A (1...,972,13724,10.77,321
2,56. Kamus Lampung-Indonesia (1985)-hasil-ekstr...,6144,27041,10.24,305
3,42. Kamus Bahasa Indonesia-Bahasa Sunda II (19...,6365,35806,11.32,486
4,10. Kamus Bahasa Indonesia-Dayak Deah Edisi I ...,5634,17598,9.07,300
5,33. Kamus Wolio Indonesia (1985)-hasil-ekstrak...,2746,12779,10.27,191
6,51. Kamus Bahasa Bali Kuno-Indonesia (1985)-ha...,1911,5688,9.95,122
7,17. Kamus Melayu Makasar-Indonesia (1985)-hasi...,2838,12798,10.21,143
8,24. Kamus Minangkabau-Indonesia (1985)-hasil-e...,11952,26256,9.92,312
9,87. Kamus Bahasa Indonesia-Kaidipang (A-K) (19...,878,30146,10.71,525


Jumlah kamus yang diekstraksi: 45

In [28]:
kamus_tidak_ekstraksi = "/Users/rafaeldewandaru/SEMESTER 8/Tugas Akhir/TAEkstraksiKamus/Harakan/daftar kamus yang tidak diekstrak.csv"
pandas_res = pd.read_csv(kamus_tidak_ekstraksi)

display(pandas_res)

,nama_kamus,jumlah_bold,jumlah_italic,mean_ukuran_char,banyak_halaman
0,13. Kamus Bahasa Indonesia-Bahasa Sunda I (199...,317,22602,10.55,408
1,50. Kamus Indonesia-Jawa Kuno (1992)-hasil-eks...,13,3312,9.59,172
2,37. Kamus Bahasa Indonesia Bakumpai II (1995)-...,42,22995,10.64,250
3,74. Kamus Bahasa Melayu Bangka-Indonesia (2018...,3627,0,11.98,156
4,79. Kamus Bahasa Jawa-Bahasa Indonesia II (199...,13,7670,10.89,415
5,25. Kamus Bahasa Indonesia Jambi L-Z (1997)-ha...,136,32818,11.26,332
6,48. Kamus Muna-Indonesia (1985)-hasil-ekstraks...,124,10311,9.78,140
7,11. Kamus Suwawa-Indonesia (1985)-hasil-ekstra...,4,34930,9.41,335
8,67. Kamus Aceh Indonesia (A-L) (1985)-hasil-ek...,112,48446,9.66,560
9,94. Kamus Bahasa Madura-Indonesia (1977)-hasil...,126,14027,7.90,208


Jumlah kamus yang tidak diekstraksi: 13

#### Perbandingan dengan korpus entri yang dibentuk Harakan

In [ ]:
csv_path = "/Users/rafaeldewandaru/SEMESTER 8/Tugas Akhir/TAEkstraksiKamus/Harakan/daftar_kamus_ekstraks.csv"
folder_path = "/Users/rafaeldewandaru/SEMESTER 8/Tugas Akhir/TAEkstraksiKamus/Ekstraksi/0. Korpus Entri"

df = pd.read_csv(csv_path)

csv_map = defaultdict(list)
folder_map = defaultdict(list)

for name in df["nama_kamus"]:
    match = re.match(r"\d+", str(name))
    if match:
        num = int(match.group())
        csv_map[num].append(name)

for f in os.listdir(folder_path):
    match = re.match(r"\d+", f)
    if match:
        num = int(match.group())
        folder_map[num].append(f)

csv_numbers = set(csv_map.keys())
folder_numbers = set(folder_map.keys())

print("Terdapat di list ektraksi kita tetapi tidak terdapat di folder korpus entri hasil ekstraksi Harakan:\n")
for num in sorted(csv_numbers - folder_numbers):
    for name in csv_map[num]:
        print(name)

print("\nTerdapat di folder korpus entri hasil ekstraksi Harakan tetapi tidak dalam list kita:\n")
for num in sorted(folder_numbers - csv_numbers):
    for name in folder_map[num]:
        print(name)

Terdapat di list ektraksi kita tetapi tidak terdapatdi folder korpus entri hasil ekstraksi Harakan:

8. Kamus Indonesia-Angkola (1995)-hasil-ekstraksi.csv
9. Kamus Manado-Indonesia (1985)-hasil-ekstraksi.csv
14. Kamus Bahasa Indonesia-Bahasa Minangkabau II (1994)-hasil-ekstraksi.csv
15. Kamus Bahasa Indonesia-Pasir (1997)-hasil-ekstraksi.csv
16. Kamus Bahasa Indonesia Karo A-K (1998)-hasil-ekstraksi.csv

Terdapat di folder korpus entri hasil ekstraksi Harakan tetapi tidak dalam list kita:

54. Kamus Bahasa Indonesia Mentawai (1998)-hasil-ekstraksi-one_entry_from_JSON-font-posisi.csv
84. Kamus Bahasa Biak - Indonesia (1977) -hasil-ekstraksi-one_entry_from_JSON-font-posisi.csv
89. Kamus Dwibahasa Bahasa Mooi-Bahasa Indonesia (2017)-hasil-ekstraksi-one_entry_from_JSON-font-posisi.csv


Sebagai perbadingan, jumlah output korpus entri yang berhasil dibentuk dari Harakan adalah sebanyak 43 kamus

### Kesimpulan